In [19]:
import ROOT


In [20]:
# CREATE THE MODEL

e =  ROOT.RooRealVar("e", "Neutrino Energy", 0.5, 14, "GeV")
e.setBins(27)

# -- read minos_2013_data.dat --

data = ROOT.RooDataSet.read("minos_2013_data.dat", e, "v")

# the default object name assigned by RooDataSet.read(..) is "dataset"
# set a more specific name to import into the RooWorkspace without ambiguity
data.SetName("data")
data.Print()

# -- read minos_2013_unbinned.dat --

mc_noosc = ROOT.RooDataSet.read("minos_2013_unbinned.dat", e, "v")
mc_noosc.Print()

# Create a binned dataset

# RooDataSet.read appends a blinding state category ("blindState").
# Remove the variable "blindState" before importing into the RooWorkspace.
dd = mc_noosc.reduce(ROOT.RooArgSet(e))
h_func_noosc = dd.binnedClone()

h_func_noosc.Print()

func_noosc = ROOT.RooHistFunc("func_noosc", "No oscillation", e, h_func_noosc, 2)

# -- define the model --

dm2 = ROOT.RooRealVar("dm2", "|#Delta m^{2}|", 2, 0.01, 10, "10^{-3} eV/c^{2}")
mixing = ROOT.RooRealVar("mixing", "sin^{2}(2#theta)", 0.90, 0.00001, 1)
L = ROOT.RooRealVar("L", "L", 730)

prob_osc = ROOT.RooFormulaVar("prob_osc",
                            "(1-mixing*pow(sin(1.27*0.001*dm2*730./e),2))",
                            ROOT.RooArgSet(mixing, dm2, e))


# model (shape of the oscillated neutrino signal - without bkg which is negligible)

model = ROOT.RooGenericPdf("model", "model", "@0*@1", 
                        ROOT.RooArgSet(prob_osc, func_noosc))

# -- fit the model --

res = model.fitTo(data, ROOT.RooFit.Save(),
                    ROOT.RooFit.PrintLevel(-2), ROOT.RooFit.Verbose(False))
# -- import into a RooWorksapce

w = ROOT.RooWorkspace()

w.Import(model)
w.Import(data)

w.Import(mc_noosc)

res.Print()



[#1] INFO:DataHandling -- RooDataSet::read: reading file minos_2013_data.dat
[#1] INFO:DataHandling -- RooDataSet::read: read 3470 events (ignored 0 out of range events)
RooDataSet::data[e,blindState] = 3470 entries
[#1] INFO:DataHandling -- RooDataSet::read: reading file minos_2013_unbinned.dat
[#1] INFO:DataHandling -- RooDataSet::read: read 48295 events (ignored 0 out of range events)
RooDataSet::dataset[e,blindState] = 48295 entries
RooDataHist::dataset_binned[e] = 27 bins (4.83e+04 weights)
[#1] INFO:Fitting -- RooAbsPdf::fitTo(model_over_model_Int[e]) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- Creation of NLL object took 256.811 μs
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_model_over_model_Int[e]_data) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
[#1] INFO:NumericIntegration -- RooRealIntegral::init(model_Int[

In [21]:
# PLOT THE MODEL

e = w.var('e')
dm2 = w.var('dm2')
mixing = w.var('mixing')
model = w.pdf('model')
data = w.data('data')

fitted_dm2 = dm2.getVal()
fitted_mix = mixing.getVal()

# -- draw a plot --

# create the plot frame

# the range of e is (0.5, 14)
# force the range from 0 to 14 to improve the plot readibility
f = e.frame(ROOT.RooFit.Range(0., 14.))

data.plotOn(f)
# prob_osc.plotOn(f, LineColor(kGreen), Normalization(500));
# pdf_noosc.plotOn(f, LineColor(kGray), RooFit::Normalization(1.3));
# func_noosc.plotOn(f, LineColor(kOrange), RooFit::Normalization(1.0965));

# -- method B --- 
dm2.setVal(0.01)
mixing.setVal(0.0001)
model.plotOn(f, ROOT.RooFit.LineColor(ROOT.kOrange),
            ROOT.RooFit.Normalization(4500, ROOT.RooAbsReal.NumEvent))
dm2.setVal(fitted_dm2)
mixing.setVal(fitted_mix)

model.plotOn(f, ROOT.RooFit.LineColor(ROOT.kRed))
model.paramOn(f)

c = ROOT.TCanvas()
f.Draw()
c.Draw()


[#1] INFO:NumericIntegration -- RooRealIntegral::init(model_Int[e]) using numeric integrator RooIntegrator1D to calculate Int(e)
[#1] INFO:NumericIntegration -- RooRealIntegral::init(model_Int[e]) using numeric integrator RooIntegrator1D to calculate Int(e)


In [22]:
# ANALYZE THE MODEL

model = w.pdf("model")
data  = w.data("data")

dm2 = w.var("dm2")
mix = w.var("mixing")

# Construct unbinned likelihood of model w.r.t. data
nll = model.createNLL(data)

# Create MINUIT interface object
# RooMinuit m(*nll); obsolete starting from ROOT v6.30
m = ROOT.RooMinimizer(nll)

# Activate verbose logging of MINUIT parameter space stepping
m.setVerbose(True)

# Call MIGRAD to minimize the likelihood
m.migrad()

# Print values of all parameters, that reflect values (and error estimates)
# that are back propagated from MINUIT
print("=== MIGRAD ===")
# model.getParameters(x)->Print("s");
dm2.Print()
mix.Print()

# Disable verbose logging
m.setVerbose(False)

# Run HESSE to calculate errors from d2L/dp2
m.hesse()

# Print value (and error) of dm2 parameter, that reflects
# value and error back propagated from MINUIT
print("=== HESSE ===")
dm2.Print()
mix.Print()

# Run MINOS on dm2 parameter only
m.minos(dm2)

# Print value (and error) of dm2 parameter, that reflects
# value and error back propagated from MINUIT
print("=== MINOS ===")
dm2.Print()
mix.Print()

"""
c_tmp = ROOT.TCanvas()
pll = nll.createProfile(ROOT.RooArgSet(dm2))

frame1 = dm2.frame(ROOT.RooFit.Title("Likelihood function"))
nll.plotOn(frame1, ROOT.RooFit.ShiftToZero())
pll.plotOn(frame1, ROOT.RooFit.LineColor(ROOT.kRed))
frame1.SetMinimum(0)
frame1.SetMaximum(8)
frame1.Draw()

ROOT.gPad.SaveAs("ex_22.png")
"""

# S a v i n g   r e s u l t s ,   c o n t o u r   p l o t s
# ---------------------------------------------------------

# Save a snapshot of the fit result. This object contains the initial
# fit parameters, the final fit parameters, the complete correlation
# matrix, the EDM, the minimized FCN , the last MINUIT status code and
# the number of times the RooFit function object has indicated evaluation
# problems (e.g. zero probabilities during likelihood evaluation)
r = m.save()

# Make contour plot of mx vs sx at 2.3 (68%), 4.61 (90%)
# RooPlot* frame = m.contour(*mix, *dm2, 2.3, 4.61);  # NO DIREI CHE BISOGNA METTERE LE SIGMA

frame = m.contour(mix, dm2, 1, 2, 3)
frame.SetTitle("RooMinuit contour plot")

# Print the fit result snapshot
r.Print("v")

c = ROOT.TCanvas("my_minos", "my_minos", 600, 600)
ROOT.gPad.SetLeftMargin(0.15)
frame.GetYaxis().SetTitleOffset(1.4)

frame.GetYaxis().SetRangeUser(1.7, 3.5)
frame.GetXaxis().SetRangeUser(0.70, 1.)

frame.Draw()
c.Draw()

ROOT.gPad.SaveAs("my_minos.png")



=== MIGRAD ===
=== HESSE ===
=== MINOS ===
[#1] INFO:Fitting -- RooAbsPdf::fitTo(model_over_model_Int[e]) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- Creation of NLL object took 388.912 μs
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_model_over_model_Int[e]_data) Summation contains a RooNLLVar, using its error level
Minuit2Minimizer: Minimize with max-calls 1000 convergence for edm < 1 strategy 1
[#1] INFO:NumericIntegration -- RooRealIntegral::init(model_Int[e]) using numeric integrator RooIntegrator1D to calculate Int(e)

prevFCN = 8393.449978  dm2=2.348, 
prevFCN = 8393.452864  dm2=2.339, 
prevFCN = 8393.452827  dm2=2.343, mixing=0.8815, 
prevFCN = 8393.452882  mixing=0.8783, 
prevFCN = 8393.452858  dm2=2.343, mixing=0.8799, 
prevFCN = 8393.449978  dm2=2.343, mixing=0.8799, 
prevFCN = 8393.449978  dm2=2.347, 
prevFCN = 8393.451982  dm2=2.34, 
prevFCN = 8393.451976  dm2=2.343, mixing=0.8812, 
prevFCN = 8393.451978  mix

Info in <Minuit2>: MnSeedGenerator Computing seed using NumericalGradient calculator
Info in <Minuit2>: MnSeedGenerator Evaluated function and gradient in 1.33814 ms
Info in <Minuit2>: MnSeedGenerator Initial state: FCN =       8393.449978 Edm =   4.299724423e-08 NCalls =      5
Info in <Minuit2>: MnSeedGenerator Initial state  
  Minimum value : 8393.449978
  Edm           : 4.299724423e-08
  Internal parameters:	[    -0.5619602942     0.8629831215]	
  Internal gradient  :	[    0.01667343681   0.002456402114]	
  Internal covariance matrix:
[[  0.00042905921              0]
 [              0   0.0087354641]]]
Info in <Minuit2>: VariableMetricBuilder Start iterating until Edm is < 0.001 with call limit = 1000
Info in <Minuit2>: VariableMetricBuilder    0 - FCN =       8393.449978 Edm =   4.299724423e-08 NCalls =      5
Info in <Minuit2>: VariableMetricBuilder    1 - FCN =       8393.449978 Edm =   1.060576707e-09 NCalls =     11
Info in <Minuit2>: MnHesse Done after 2.947 ms
Info in <Mi